In [1]:
import torch
import torch.nn as nn


class StockPctChangeLSTM(nn.Module):
    def __init__(
        self,
        input_size=5,       # OHLCV — expand later when adding indicators
        hidden_size=64,
        num_layers=2,
        dropout=0.2
    ):
        super(StockPctChangeLSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.bn      = nn.BatchNorm1d(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 2)  # [price_pct_change, volume_pct_change]

    def forward(self, x):
        out, _  = self.lstm(x)
        out     = out[:, -1, :]     # last timestep (batch, hidden_size)
        out     = self.bn(out)
        out     = self.dropout(out)
        out     = self.fc(out)      # (batch, 2)
        return out


# Loss & Optimizer
# criterion = nn.HuberLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)